In [9]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Tabellen-Kopfzeile erstellen
header_layout = widgets.Layout(width='110px', justify_content='center')
header = widgets.HBox([
    widgets.HTML("<b>#</b>", layout=widgets.Layout(width='30px')),
    widgets.HTML("<b>Amplitude</b>", layout=header_layout),
    widgets.HTML("<b>Frequenz (Hz)</b>", layout=header_layout),
    widgets.HTML("<b>Phase (°)</b>", layout=header_layout)
])

# 2. 10 Eingabezeilen generieren
zeilen = []
eingabefelder = []
startwerte = [(1.0, 5.0, 0.0), (0.5, 15.0, 90.0)]

for i in range(10):
    amp_val, freq_val, phase_val = startwerte[i] if i < len(startwerte) else (0.0, 0.0, 0.0)
    lbl = widgets.Label(f"{i+1}.", layout=widgets.Layout(width='30px'))
    amp = widgets.FloatText(value=amp_val, step=0.1, layout=widgets.Layout(width='110px'))
    freq = widgets.FloatText(value=freq_val, step=1.0, layout=widgets.Layout(width='110px'))
    phase = widgets.FloatText(value=phase_val, step=15.0, layout=widgets.Layout(width='110px'))
    eingabefelder.append((amp, freq, phase))
    zeilen.append(widgets.HBox([lbl, amp, freq, phase]))

tabelle = widgets.VBox([header] + zeilen)

# 3. Steuerungs-Elemente & feiner Zeit-Slider
max_freq_input = widgets.FloatText(
    value=60.0, min=5.0, max=5000.0, step=10.0,
    description='Max. Frequenz (Hz):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

schrittweite = 0.001
anzahl_schritte = int(round(1.0 / schrittweite))

play_widget = widgets.Play(value=0, min=0, max=anzahl_schritte, step=1, interval=30, description="Animation")
zeit_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=schrittweite,
    description='Zeit t (s):', readout_format='.3f', continuous_update=True,
    style={'description_width': 'initial'}, layout=widgets.Layout(width='520px')
)

def sync_play_zu_slider(change):
    ziel_zeit = round(change['new'] * schrittweite, 4)
    if abs(zeit_slider.value - ziel_zeit) > 1e-6:
        zeit_slider.value = ziel_zeit

def sync_slider_zu_play(change):
    ziel_step = int(round(change['new'] / schrittweite))
    if play_widget.value != ziel_step:
        play_widget.value = ziel_step

play_widget.observe(sync_play_zu_slider, names='value')
zeit_slider.observe(sync_slider_zu_play, names='value')

btn_zeichnen = widgets.Button(description="Signal & FFT berechnen", button_style='primary', icon='play')
btn_leeren = widgets.Button(description="Tabelle leeren", button_style='warning', icon='trash')
output = widgets.Output()

plot_state = {
    'fig': None, 'ax4': None, 'aktive_komponenten': [],
    'zeit_linie': None, 'zeit_punkt': None, 'zeit_legend': None,
    'zeiger_pfeile': [], 'zeiger_kreise': [], 'spitze_punkt': None
}
def aktualisiere_animation(change=None):
    if plot_state['fig'] is None:
        return
    t_aktuell = zeit_slider.value
    aktive = plot_state['aktive_komponenten']
    y_aktuell = sum(a * np.sin(2 * np.pi * f * t_aktuell + p_rad) for a, f, _, p_rad in aktive)
    plot_state['zeit_linie'].set_xdata([t_aktuell, t_aktuell])
    plot_state['zeit_punkt'].set_data([t_aktuell], [y_aktuell])
    if plot_state['zeit_legend'] is not None and len(plot_state['zeit_legend'].get_texts()) > 1:
        plot_state['zeit_legend'].get_texts()[1].set_text(f'Momentanwert ({y_aktuell:.2f})')

    start_re, start_im = 0.0, 0.0
    for idx, (a, f, _, p_rad) in enumerate(aktive):
        winkel = 2 * np.pi * f * t_aktuell + p_rad
        d_re = a * np.cos(winkel)
        d_im = a * np.sin(winkel)
        plot_state['zeiger_kreise'][idx].center = (start_re, start_im)
        pfeil = plot_state['zeiger_pfeile'][idx]
        pfeil.set_offsets([[start_re, start_im]])
        pfeil.set_UVC(d_re, d_im)
        start_re += d_re
        start_im += d_im

    if plot_state['spitze_punkt'] is not None:
        if aktive:
            plot_state['spitze_punkt'].set_data([start_re], [start_im])
        else:
            plot_state['spitze_punkt'].set_data([], [])

    plot_state['ax4'].set_title(
        f'4. Komplexe Ebene: Hintereinandergehängte Zeiger & Bahnkurve bei t = {t_aktuell:.3f} s',
        fontsize=11, fontweight='bold'
    )
    with output:
        clear_output(wait=True)
        display(plot_state['fig'])

def berechne_und_zeichne(b=None):
    plt.close('all')
    max_f = max(5.0, max_freq_input.value)
    t_aktuell = zeit_slider.value
    fs = max(1000, int(max_f * 10))
    t = np.linspace(0, 1, fs, endpoint=False)
    signal = np.zeros_like(t)
    aktive_komponenten = []

    for amp_w, freq_w, phase_w in eingabefelder:
        a = amp_w.value
        f = freq_w.value
        p_deg = phase_w.value
        p_rad = np.deg2rad(p_deg)
        if a != 0 and f > 0:
            signal += a * np.sin(2 * np.pi * f * t + p_rad)
            aktive_komponenten.append((a, f, p_deg, p_rad))

    plot_state['aktive_komponenten'] = aktive_komponenten
    fft_werte = np.fft.rfft(signal)
    frequenzen = np.fft.rfftfreq(len(t), d=1/fs)
    amplituden = (2.0 / len(t)) * np.abs(fft_werte)
    amplituden_gefiltert = np.where(amplituden > 0.01, amplituden, np.nan)
    phasen = (np.angle(fft_werte, deg=True) + 90.0 + 180) % 360 - 180
    phasen_gefiltert = np.where(amplituden > 0.01, phasen, np.nan)

    fig, (ax1, ax2, ax3, ax4) = plt.subplots(
        4, 1, figsize=(10, 16), gridspec_kw={'height_ratios': [1, 1, 1, 2.3]}
    )
    plt.subplots_adjust(hspace=0.45)

    # Diagramm 1: Zeitbereich
    ax1.plot(t, signal, color='#1f77b4', lw=1.5, label='Summensignal y(t)')
    y_aktuell = sum(a * np.sin(2 * np.pi * f * t_aktuell + p_rad) for a, f, _, p_rad in aktive_komponenten)
    zeit_linie = ax1.axvline(t_aktuell, color='black', linestyle='--', alpha=0.6)
    zeit_punkt, = ax1.plot([t_aktuell], [y_aktuell], 'ro', markersize=7, label=f'Momentanwert ({y_aktuell:.2f})')
    ax1.set_title('1. Zeitbereich: Summiertes Signal (Imaginärteil der Zeigerspitze)', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Zeit (Sekunden)')
    ax1.set_ylabel('Amplitude')
    ax1.set_xlim(0, 1)
    ax1.grid(True, alpha=0.3)
    zeit_legend = ax1.legend(loc='upper right', fontsize=8.5)

    # Diagramm 2: Amplitudenspektrum
    if not np.all(np.isnan(amplituden_gefiltert)):
        _, stemlines_amp, _ = ax2.stem(frequenzen, amplituden_gefiltert, linefmt='#d62728', markerfmt='ro', basefmt=' ')
        plt.setp(stemlines_amp, linewidth=2)
    ax2.set_title('2. Frequenzbereich: Amplitudenspektrum (nur aktive Frequenzen)', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Frequenz (Hz)')
    ax2.set_ylabel('Amplitude')
    ax2.set_xlim(0, max_f)
    max_amp = np.nanmax(amplituden_gefiltert) if not np.all(np.isnan(amplituden_gefiltert)) else 1.0
    ax2.set_ylim(0, max_amp * 1.2)
    ax2.grid(True, alpha=0.3)

    # Diagramm 3: Phasenspektrum
    if not np.all(np.isnan(phasen_gefiltert)):
        _, stemlines_ph, _ = ax3.stem(frequenzen, phasen_gefiltert, linefmt='#2ca02c', markerfmt='go', basefmt=' ')
        plt.setp(stemlines_ph, linewidth=2)
    ax3.set_title('3. Frequenzbereich: Phasenspektrum (nur aktive Frequenzen)', fontsize=11, fontweight='bold')
    ax3.set_xlabel('Frequenz (Hz)')
    ax3.set_ylabel('Phase (°)')
    ax3.set_xlim(0, max_f)
    ax3.set_ylim(-195, 195)
    ax3.set_yticks([-180, -90, 0, 90, 180])
    ax3.grid(True, alpha=0.3)

    # Diagramm 4: Komplexe Ebene mit Bahnkurve
    ax4.axhline(0, color='black', lw=0.9, alpha=0.5)
    ax4.axvline(0, color='black', lw=0.9, alpha=0.5)
    ax4.set_aspect('equal', adjustable='box')
    farben = plt.cm.tab10.colors
    gesamt_radius = sum(abs(a) for a, _, _, _ in aktive_komponenten) if aktive_komponenten else 1.0
    zeiger_pfeile = []
    zeiger_kreise = []

    if aktive_komponenten:
        t_spur = np.linspace(0, 1, 2500, endpoint=True)
        spur_re = np.zeros_like(t_spur)
        spur_im = np.zeros_like(t_spur)
        for a, f, _, p_rad in aktive_komponenten:
            spur_re += a * np.cos(2 * np.pi * f * t_spur + p_rad)
            spur_im += a * np.sin(2 * np.pi * f * t_spur + p_rad)
        ax4.plot(spur_re, spur_im, color='#444444', lw=1.3, alpha=0.55, label='Bahnkurve (Spur)')

        start_re, start_im = 0.0, 0.0
        for idx, (a, f, p_deg, p_rad) in enumerate(aktive_komponenten):
            farbe = farben[idx % len(farben)]
            winkel = 2 * np.pi * f * t_aktuell + p_rad
            d_re = a * np.cos(winkel)
            d_im = a * np.sin(winkel)
            kreis = plt.Circle((start_re, start_im), abs(a), color=farbe, fill=False, linestyle='--', alpha=0.35)
            ax4.add_patch(kreis)
            zeiger_kreise.append(kreis)
            pfeil = ax4.quiver(
                start_re, start_im, d_re, d_im,
                angles='xy', scale_units='xy', scale=1,
                color=farbe, width=0.005, label=f'{f} Hz: A={a}, φ={p_deg}°'
            )
            zeiger_pfeile.append(pfeil)
            start_re += d_re
            start_im += d_im
        spitze_punkt, = ax4.plot([start_re], [start_im], 'ro', markersize=7, label='Aktuelle Zeigerspitze')
    else:
        spitze_punkt, = ax4.plot([], [], 'ro', markersize=7)

    limit = max(1.0, gesamt_radius * 1.12)
    ax4.set_xlim(-limit, limit)
    ax4.set_ylim(-limit, limit)
    ax4.set_title(
        f'4. Komplexe Ebene: Hintereinandergehängte Zeiger & Bahnkurve bei t = {t_aktuell:.3f} s',
        fontsize=11, fontweight='bold'
    )
    ax4.set_xlabel('Realteil (Re)')
    ax4.set_ylabel('Imaginärteil (Im) → Amplitude')
    ax4.grid(True, alpha=0.3)
    if aktive_komponenten:
        ax4.legend(bbox_to_anchor=(1.03, 1), loc='upper left', borderaxespad=0., fontsize=9.5)

    plot_state.update({
        'fig': fig, 'ax4': ax4, 'zeit_linie': zeit_linie, 'zeit_punkt': zeit_punkt,
        'zeit_legend': zeit_legend, 'zeiger_pfeile': zeiger_pfeile,
        'zeiger_kreise': zeiger_kreise, 'spitze_punkt': spitze_punkt
    })
    plt.close(fig)
    with output:
        clear_output(wait=True)
        display(fig)

def tabelle_leeren(b):
    play_widget.playing = False
    for amp_w, freq_w, phase_w in eingabefelder:
        amp_w.value = 0.0
        freq_w.value = 0.0
        phase_w.value = 0.0
    zeit_slider.value = 0.0
    berechne_und_zeichne()

btn_zeichnen.on_click(berechne_und_zeichne)
btn_leeren.on_click(tabelle_leeren)
zeit_slider.observe(aktualisiere_animation, names='value')

steuerleiste = widgets.HBox([max_freq_input, btn_zeichnen, btn_leeren], layout=widgets.Layout(margin='10px 0px'))
animations_leiste = widgets.HBox([play_widget, zeit_slider], layout=widgets.Layout(margin='10px 0px 25px 0px', align_items='center'))

display(tabelle, steuerleiste, output, animations_leiste)
berechne_und_zeichne()


Output()